# Notebook 00 — Introduction et contexte

> **Série PRJ2025_773 — Costing hiérarchique & données rares**  
> Ce notebook est le point d'entrée. Il pose le problème, génère les données synthétiques partagées entre tous les notebooks, et donne une carte mentale de la série.

---

## Le problème industriel

Tu es Responsable Données dans une entreprise qui fabrique des **onduleurs et UPS** (groupes de continuité électrique).  
Le catalogue compte **45 familles** de produits, chacune déclinée en **variantes** (puissance, connectique, certifications…).

**Le challenge :** estimer le coût de fabrication de chaque variante pour alimenter les offres commerciales.  
- Variantes matures (> 12 commandes) → données suffisantes, modèle individuel viable  
- **Variantes rares (< 6 commandes)** → données insuffisantes, risque d'over-fitting massif  

La question centrale : **comment tirer parti de la structure hiérarchique** (variante ⊂ famille ⊂ gamme) pour produire des estimations fiables même quand les données manquent ?

---

## Ordre de lecture recommandé : pragmatique d'abord

La démarche suit une logique **du plus simple à valider au plus riche** :

1. **Diagnostiquer** le problème — comprendre pourquoi les données rares posent problème (NB 01)
2. **Résoudre** avec Transfer Learning — baseline solide, cold start immédiat, sklearn standard (NB 04) ⭐ *commencer ici*
3. **Expliquer** avec Mixed Effects — couche lisible pour le CODIR, si la performance valide (NB 02)
4. **Affiner** avec le bayésien — seulement si des intervalles de confiance formels sont nécessaires (NB 03)
5. **Valider** correctement à chaque étape (NB 05)
6. **Industrialiser** avec le pipeline cold→mature (NB 06)

| # fichier | Notebook | Concept clé | Quand l'utiliser |
|-----------|----------|-------------|-----------------|
| 00 | Introduction & données | Génération des données synthétiques | Toujours en premier |
| 01 | Diagnostic variantes rares | Biais-variance, détection over-fitting | Toujours |
| **04** | **Transfer Learning ⭐** | **Approche C — cold start, fine-tuning** | **Point de départ recommandé** |
| 02 | Mixed Effects Model | Approche B — effets fixes + aléatoires | Si explicabilité CODIR nécessaire |
| 03 | Hierarchical Bayesian | Approche A — shrinkage bayésien | Si intervalles de confiance formels nécessaires |
| 05 | Protocole de validation | LOO-CV, métriques asymétriques | À chaque étape |
| 06 | Intégration expert & cold start | Priors informés, pipeline complet | Industrialisation |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = Path('data')
DATA_PATH.mkdir(exist_ok=True)

## Génération des données synthétiques

On simule un catalogue réaliste :
- **5 gammes** → **15 familles** (3 par gamme) → **120 variantes** (~8 par famille)
- Pour chaque variante : `n_obs` commandes historiques, avec des **features de coût** et le **coût réel**

### Modèle de génération des données

Le coût d'une commande `t` pour la variante `v` dans la famille `f` est :

$$\text{Coût}_{vt} = \alpha_{\text{gamme}(f)} + \beta_f + u_v + \gamma_1 \cdot \text{Composants}_{vt} + \gamma_2 \cdot \text{MO}_{vt} + \gamma_3 \cdot \text{Énergie}_{vt} + \varepsilon_{vt}$$

Avec :
- $\alpha$ : effet gamme (niveau de coût global)
- $\beta_f$ : effet famille (spécificité technologique)
- $u_v \sim \mathcal{N}(0, \sigma_v^2)$ : effet variante (déviation propre)
- $\varepsilon_{vt} \sim \mathcal{N}(0, \sigma_{\varepsilon}^2)$ : bruit résiduel

In [ ]:
# --- Paramètres de la hiérarchie ---
N_GAMMES   = 5
N_FAMILLES = 15   # 3 par gamme
N_VARIANTES = 120  # ~8 par famille

gamme_names   = [f'Gamme_{g}' for g in 'ABCDE']
famille_names = [f'FAM_{g}{i}' for g in 'ABCDE' for i in range(1, 4)]
variante_names = [f'VAR_{f}_{i:02d}' for f in famille_names for i in range(1, 9)]

# Effets de niveaux (en €)
alpha_gamme   = {g: base for g, base in zip(gamme_names, [800, 1200, 600, 1500, 950])}
beta_famille  = {f: np.random.normal(0, 150) for f in famille_names}
u_variante    = {v: np.random.normal(0, 60)  for v in variante_names}

# Coefficients des cost drivers
gamma = {'composants': 0.55, 'mo': 0.25, 'energie': 0.10}

print(f"Hiérarchie : {N_GAMMES} gammes → {N_FAMILLES} familles → {N_VARIANTES} variantes")

In [ ]:
# --- Distribution du nombre d'observations par variante ---
# On crée intentionnellement une distribution hétérogène
def sample_n_obs(n_variants: int) -> list:
    """Simule la répartition réelle : majorité de variantes rares."""
    rng = np.random.default_rng(42)
    # ~30% de variantes rares (1-5 obs), ~40% intermédiaires, ~30% matures
    categories = rng.choice(['rare', 'intermediate', 'mature'],
                             size=n_variants, p=[0.30, 0.40, 0.30])
    n_obs = []
    for cat in categories:
        if cat == 'rare':
            n_obs.append(rng.integers(1, 6))
        elif cat == 'intermediate':
            n_obs.append(rng.integers(6, 25))
        else:
            n_obs.append(rng.integers(25, 80))
    return n_obs

n_obs_per_variant = sample_n_obs(N_VARIANTES)

print(f"Répartition des variantes :")
print(f"  Rares     (1-5 obs)  : {sum(1 for n in n_obs_per_variant if n < 6):>3d} variantes")
print(f"  Interméd. (6-24 obs) : {sum(1 for n in n_obs_per_variant if 6 <= n < 25):>3d} variantes")
print(f"  Matures   (≥25 obs)  : {sum(1 for n in n_obs_per_variant if n >= 25):>3d} variantes")

In [ ]:
# --- Génération du dataset complet ---
rows = []
rng  = np.random.default_rng(99)

for idx, variante in enumerate(variante_names):
    famille = '_'.join(variante.split('_')[:2])   # ex: FAR_AB1
    # Reconstruction correcte : FAM_A1, FAM_A2 …
    fam_parts = variante.split('_')               # ['VAR', 'FAM', 'A1', '01']
    famille   = f"{fam_parts[1]}_{fam_parts[2]}" # FAM_A1
    gamme     = 'Gamme_' + fam_parts[2][0]        # Gamme_A

    n = n_obs_per_variant[idx]
    base_cost = alpha_gamme[gamme] + beta_famille[famille] + u_variante[variante]

    for t in range(n):
        composants = rng.normal(base_cost * 0.5, base_cost * 0.05)
        mo         = rng.normal(base_cost * 0.2, base_cost * 0.03)
        energie    = rng.normal(base_cost * 0.1, base_cost * 0.02)
        volume     = rng.integers(1, 50)
        noise      = rng.normal(0, 50)

        cout = (gamma['composants'] * composants +
                gamma['mo']         * mo         +
                gamma['energie']    * energie    +
                0.02 * volume + noise + base_cost * 0.1)

        rows.append({
            'variante'  : variante,
            'famille'   : famille,
            'gamme'     : gamme,
            'n_obs_variante': n,
            'composants': composants,
            'mo'        : mo,
            'energie'   : energie,
            'volume'    : volume,
            'cout'      : max(cout, 100),  # coût plancher à 100€
            'obs_index' : t,
        })

df = pd.DataFrame(rows)
df.to_parquet(DATA_PATH / 'dataset_industriel.parquet', index=False)
print(f"Dataset généré : {len(df):,} lignes, {df['variante'].nunique()} variantes")
df.head()

In [ ]:
# --- Visualisation de la distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution du nombre d'obs par variante
ax = axes[0]
colors = ['#e74c3c' if n < 6 else '#f39c12' if n < 25 else '#27ae60'
          for n in n_obs_per_variant]
ax.bar(range(N_VARIANTES), sorted(n_obs_per_variant), color=sorted(colors, key=lambda c: {'#e74c3c':0,'#f39c12':1,'#27ae60':2}[c]))
ax.axhline(6,  color='#e74c3c', linestyle='--', label='Seuil rare (6 obs)')
ax.axhline(25, color='#27ae60', linestyle='--', label='Seuil mature (25 obs)')
ax.set_xlabel('Variante (triée par n_obs)')
ax.set_ylabel('Nombre d\'observations')
ax.set_title('Distribution des observations par variante')
ax.legend()

# Distribution des coûts par catégorie
ax = axes[1]
df['categorie'] = pd.cut(df['n_obs_variante'], bins=[0,5,24,200],
                          labels=['Rare (≤5)', 'Interméd. (6-24)', 'Mature (≥25)'])
for cat, color in [('Rare (≤5)','#e74c3c'), ('Interméd. (6-24)','#f39c12'), ('Mature (≥25)','#27ae60')]:
    data = df[df['categorie'] == cat]['cout']
    ax.hist(data, bins=30, alpha=0.6, color=color, label=f'{cat} (n={len(data)})')
ax.set_xlabel('Coût (€)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des coûts par catégorie de variante')
ax.legend()

plt.tight_layout()
plt.savefig(DATA_PATH / 'fig_distribution_dataset.png', dpi=150)
plt.show()
print("Figure sauvegardée.")

## Résumé

- Le dataset est sauvegardé dans `data/dataset_industriel.parquet`
- Il contient une hiérarchie à **3 niveaux** : gamme → famille → variante
- ~30% des variantes ont **< 6 observations** — c'est la zone à risque
- Les notebooks suivants importent ce fichier directement

**→ Notebook suivant : [01_diagnostic_variantes_rares.ipynb](01_diagnostic_variantes_rares.ipynb)**